# N4_Revised_Forecast
## Treasury Module

**CFOPackV001: Treasury Decision Workshop**

---

© Copyright 2026 Professor Vinaya Sathyanarayana

All rights reserved. This notebook is provided as part of the CFOPackV001 Treasury Decision Workshop.
Attribution required: Please retain this copyright notice and credit Professor Vinaya Sathyanarayana in any derivative work.

**Contact:** vinallcontact@gmail.com  
**GitHub:** https://github.com/VinayaSharada/KateelLearningDemosToStudents


## Learning Objectives

By the end of this module, you will be able to:

- **Understand** the business decision this notebook supports
- **Execute** the analysis workflow without errors
- **Interpret** outputs in plain English
- **Explain** the assumptions behind each calculation
- **Adapt** the code for your own data

**Estimated Time:** ** 20-40 minutes (including reading code and outputs)


## Overview### What This Notebook DoesThis notebook builds a realistic 14-day cash forecast using ML-predicted payment dates instead of due dates, showing what actually happens if customers pay like they have historically.### Why It MattersThe revised forecast is the most operationally accurate scenario:- **Reality-based:** Reflects actual customer payment patterns, not wishful thinking- **Risk quantification:** Shows true cash position and shortfall dates- **Gap analysis:** Compares realistic vs. optimistic forecasts to quantify payment delay risk- **Actionable insights:** Identifies specific days you need external funding or liquidity### What Data It Uses- `N1_validated_data.csv` – Invoice list- `N3_invoice_payment_predictions.csv` – ML-predicted payment dates- `N1_cash_flow.csv` – Daily operating outflows### What Outputs It Creates- `N4_revised_forecast.csv` – Daily cash position using realistic payment dates- `N4_gap_analysis.csv` – Day-by-day comparison: baseline vs. revised forecast

## Execute Workflow

# N4_Revised_Forecast
## Treasury Module

**CFOPackV001: Treasury Decision Workshop**

---

© Copyright 2026 Professor Vinaya Sathyanarayana

All rights reserved. This notebook is provided as part of the CFOPackV001 Treasury Decision Workshop.
Attribution required: Please retain this copyright notice and credit Professor Vinaya Sathyanarayana in any derivative work.

**Contact:** vinallcontact@gmail.com  
**GitHub:** https://github.com/VinayaSharada/KateelLearningDemosToStudents


## Learning Objectives

By the end of this module, you will be able to:

- **Understand** the business decision this notebook supports
- **Execute** the analysis workflow without errors
- **Interpret** outputs in plain English
- **Explain** the assumptions behind each calculation
- **Adapt** the code for your own data

**Estimated Time:** ** 20-40 minutes (including reading code and outputs)


In [ ]:
# ==============================================================================
# SETUP: Imports and Configuration
# ==============================================================================
# This cell imports all required libraries and configures data sources.
# No changes needed unless you want to use your own data.

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
# CONFIGURATION: Choose your data source
USE_GITHUB_DATA = True  # Set to False if you want to upload your own data
GITHUB_RAW_URL = 'https://raw.githubusercontent.com/VinayaSharada/KateelLearningDemosToStudents/main/CFOPackV001/data/synthetic'

print('✓ Imports successful')
print(f"✓ Data source: {'GitHub (synthetic)' if USE_GITHUB_DATA else 'Manual upload'}")

In [ ]:
# ==============================================================================
# LOAD DATA FROM PREVIOUS NOTEBOOKS (N1, N2, N3)
# ==============================================================================
# This cell loads outputs from previous notebooks with fallbacks

print("[[LOAD] Loading data from N1, N2, N3 outputs...")
print()
# Load baseline forecast (from N2)
try:
    baseline_forecast = pd.read_csv("../outputs/N2_baseline_forecast.csv")
print(f"[OK] Loaded baseline forecast from N2: {len(baseline_forecast)} days")
except FileNotFoundError:
    print("[WARNING] N2 baseline forecast not found, will create placeholder")
    baseline_forecast = None

# Load predictions (from N3)
try:
    predictions = pd.read_csv("../outputs/N3_invoice_payment_predictions.csv")
print(f"[OK] Loaded predictions from N3: {len(predictions)} invoices")
except FileNotFoundError:
    print("[WARNING] N3 predictions not found, will create placeholder")
    predictions = None

# Load cash flow (from N1)
try:
    cash_flow = pd.read_csv("../outputs/N1_cash_flow.csv")
print(f"[OK] Loaded cash flow from N1: {len(cash_flow)} days")
except FileNotFoundError:
    # Fallback to source data
    try:
        cash_flow = pd.read_csv(f'{GITHUB_RAW_URL}/cash_flow.csv')
print(f"[OK] Loaded cash flow from GitHub: {len(cash_flow)} days")
    except:
        print("[ERROR] Could not load cash flow")
        cash_flow = None

# Convert date columns
if baseline_forecast is not None:
    baseline_forecast['date'] = pd.to_datetime(baseline_forecast['date'])
if predictions is not None:
    predictions['due_date'] = pd.to_datetime(predictions['due_date'])
    predictions['predicted_payment_date'] = pd.to_datetime(predictions['predicted_payment_date'])
if cash_flow is not None:
    cash_flow['date'] = pd.to_datetime(cash_flow['date'])
print()
# ============================================================================

In [ ]:
# ==============================================================================
# DATA LOADING: GitHub or Manual Upload
# ==============================================================================
# Define two data loading methods and use whichever matches your choice above.

def load_data_from_github():
    """Load synthetic data directly from GitHub repository.
    
    Advantages:
    - No API key required
    - Pre-validated and consistent with reference outputs
    - Fast (uses GitHub CDN)
    
    Returns: dict with keys 'invoices', 'payments', 'customers'
    """
    try:
        print('Loading data from GitHub...')
        invoices = pd.read_csv(f'{GITHUB_RAW_URL}/invoices.csv')
        payments = pd.read_csv(f'{GITHUB_RAW_URL}/payments.csv')
        customers = pd.read_csv(f'{GITHUB_RAW_URL}/customers.csv')
        
        print(f'✓ Loaded {len(invoices):,} invoices')
        print(f'✓ Loaded {len(payments):,} payments')
        print(f'✓ Loaded {len(customers):,} customers')
        return {'invoices': invoices, 'payments': payments, 'customers': customers}
    except Exception as e:
        print(f'✗ Error: {e}')
        print('  Try Option 2: Manual upload')
        return None

def load_data_from_upload():
    """Load data from files you upload manually.
    
    In Colab: Click Files panel → Upload → Select CSVs
    In Jupyter: Put CSVs in the same folder as this notebook
    
    Required files: invoices.csv, payments.csv, customers.csv
    See data/README.md for required columns.
    """
    try:
        print('Loading data from uploaded files...')
        invoices = pd.read_csv('invoices.csv')
        payments = pd.read_csv('payments.csv')
        customers = pd.read_csv('customers.csv')
        
        print(f'✓ Loaded {len(invoices):,} invoices')
        print(f'✓ Loaded {len(payments):,} payments')
        print(f'✓ Loaded {len(customers):,} customers')
        return {'invoices': invoices, 'payments': payments, 'customers': customers}
    except FileNotFoundError as e:
        print(f'✗ File not found: {e}')
        return None

# Execute data loading based on configuration above
if USE_GITHUB_DATA:
    data = load_data_from_github()
else:
    data = load_data_from_upload()

if data is None:
    print('\n⚠ Data loading failed. Check error above.')
else:
    invoices = data['invoices']
    payments = data['payments']
    customers = data['customers']
    print('\n✓ All data loaded and ready for analysis!')


## REBUILD FORECAST WITH PREDICTED PAYMENT DATES

**Purpose:** Use predicted payment dates to rebuild the forecast model.

**Code Section:** ~41 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[BANK] Building revised forecast using predicted payment dates...")
print()forecast_start = cash_flow['date'].min()forecast_end = cash_flow['date'].max()starting_cash = 5_000_000
# Get inflows using PREDICTED payment datesrevised_inflows = predictions[['invoice_id', 'predicted_payment_date', 'amount_usd']].copy()revised_inflows.columns = ['invoice_id', 'payment_date', 'payment_amount']revised_inflows = revised_inflows[revised_inflows['payment_date'] >= forecast_start]revised_inflows = revised_inflows[revised_inflows['payment_date'] <= forecast_end]print(f"Inflows (using predicted dates): {len(revised_inflows)}")
print(f"Total: ${revised_inflows['payment_amount'].sum():,.0f}")
print()
# Build day-by-day forecastdaily_revised = []for day_num in range(len(cash_flow)):    day_row = cash_flow.iloc[day_num]    day_date = day_row['date']
# Get inflows for this day (using predicted payment dates)    day_inflows = revised_inflows[revised_inflows['payment_date'] == day_date]['payment_amount'].sum()
# Get outflows    day_outflows = day_row['total_outflows']
# Calculate cash    if day_num == 0:        opening_cash = starting_cash    else:        opening_cash = daily_revised[day_num - 1]['closing_cash']    net_change = day_inflows - day_outflows    closing_cash = opening_cash + net_change    daily_revised.append({        'day': day_num + 1,        'date': day_date,        'opening_cash': opening_cash,        'inflows': day_inflows,        'outflows': day_outflows,        'net_change': net_change,        'closing_cash': max(0, closing_cash)    })revised_forecast = pd.DataFrame(daily_revised)
print("[[OK] Revised 14-day forecast complete")
print()

## GAP ANALYSIS

**Purpose:** Analyze discrepancies between original and revised forecasts.

**Code Section:** ~38 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[CHART] BASELINE vs REVISED COMPARISON")
print()
# Merge for comparisoncomparison = baseline_forecast[['day', 'date', 'closing_cash']].copy()comparison.columns = ['day', 'date', 'baseline_closing_cash']comparison['revised_closing_cash'] = revised_forecast['closing_cash'].valuescomparison['gap'] = comparison['baseline_closing_cash'] - comparison['revised_closing_cash']print("[Day-by-day gap:")
print("[-" * 100)for idx, row in comparison.iterrows():    if row['gap'] > 0:        print(f"  Day {int(row['day']):2d}: Baseline ${row['baseline_closing_cash']:>10,.0f}  "              f"Revised ${row['revised_closing_cash']:>10,.0f}  "              f"Gap ${row['gap']:>10,.0f}")
print("[-" * 100)
print()
# Key metricsbaseline_min = comparison['baseline_closing_cash'].min()revised_min = comparison['revised_closing_cash'].min()total_gap_day14 = comparison.iloc[-1]['gap']print(f"Summary Metrics:")
print(f"  Baseline minimum: ${baseline_min:,.0f} (Day {int(comparison[comparison['baseline_closing_cash'] == baseline_min]['day'].values[0])})")
print(f"  Revised minimum:  ${revised_min:,.0f} (Day {int(comparison[comparison['revised_closing_cash'] == revised_min]['day'].values[0])})")
print(f"  Gap on Day 14:    ${total_gap_day14:,.0f}")
print()
# Interpretationprint("[[WARNING]  WHAT THIS MEANS:")
print(f"   Baseline (optimistic): Assumes all invoices pay on time")
print(f"   Revised (realistic): Assumes invoices pay {predictions['predicted_days_late'].mean():.1f} days late")
print(f"   Cash gap on Day 14: ${total_gap_day14:,.0f}")
print()if total_gap_day14 > 500_000:    print(f"  [ALERT] This is a MATERIAL gap. Needs action.")elif total_gap_day14 > 200_000:    print(f"  [WARNING]  This is SIGNIFICANT. Consider mitigation strategies.")else:    print(f"  [OK] Gap is manageable with existing credit facility.")
print()

## RISK ZONES

**Purpose:** Identify and assess payment risks across customer segments.

**Code Section:** ~12 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[ CASH POSITION RISK ZONES")
print()danger_threshold = 1_000_000revised_in_danger = revised_forecast[revised_forecast['closing_cash'] < danger_threshold]if len(revised_in_danger) > 0:    print(f"[WARNING]  Days when cash falls below ${danger_threshold:,.0f}:")    for idx, row in revised_in_danger.iterrows():        print(f"    Day {int(row['day'])}: ${row['closing_cash']:,.0f}")
print()else:    print(f"[OK] Cash stays above ${danger_threshold:,.0f} throughout forecast period")
print()

## EXPORT RESULTS

**Purpose:** Save analysis outputs for stakeholder presentation.

**Code Section:** ~8 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[SAVE] Exporting revised forecast...")export_path = "../outputs/N4_revised_forecast.csv"revised_forecast.to_csv(export_path, index=False)
print(f"[OK] Exported: {export_path}")export_path = "../outputs/N4_gap_analysis.csv"comparison.to_csv(export_path, index=False)
print(f"[OK] Exported: {export_path}")
print()

## KEY INSIGHTS

**Purpose:** Execute key insights analysis.

**Code Section:** ~9 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[=" * 80)
print("[[DONE] N4 COMPLETE - Revised Forecast Built")
print("[=" * 80)
print()
print("[[INFO] Key Finding:")
print(f"  Realistic cash position is ${total_gap_day14:,.0f} LOWER than baseline on Day 14")
print()
print("[[GOAL] Next step: N5_Working_Capital_Levers.py")
print("[   Model which operational levers (collections, inventory, payables) can close this gap")

## Download Your ResultsThis notebook generated the following files. Download them to your computer:### Output Files| File Name | Description | Size | Download ||-----------|-------------|------|----------|| N4_revised_forecast.csv | Daily cash position using realistic payment predictions | ~10 KB | [Download](#) || N4_gap_analysis.csv | Comparison of baseline vs revised forecast | ~10 KB | [Download](#) |### How to Download in Colab1. Click the **Files** icon (📁) in left sidebar2. Right-click the output files folder3. Select **Download**### Where Files Are Saved- **Colab:** `/content/outputs/` (download to your computer)- **Local Jupyter:** `../outputs/` (same directory as notebook)- **Next Step:** Use these files in the next notebook### What Each File Contains- **N4_revised_forecast.csv:** Daily cash position using realistic payment predictions- **N4_gap_analysis.csv:** Comparison of baseline vs revised forecast

---

## Module Complete!

You have successfully completed this module. Your outputs are ready for the next step.

**Next Module:** Open the next notebook to continue the workshop.

**Questions or Issues?**
- Review the Learning Objectives and inline comments above
- Check `participant/GETTING_STARTED.md` for help
- Email: vinallcontact@gmail.com

---
© 2026 Professor Vinaya Sathyanarayana | CFOPackV001 Treasury Decision Workshop
